# Day 066 — Solution: Image-Processing Utility

In [ ]:
_UTILS_SRC = '"""image_utils.py — Day 066: Chainable image-processing utility.\n\nUsage:\n    from image_utils import ImageProcessor\n    img = (ImageProcessor.new(800, 600, color=(200, 220, 240))\n           .resize(400, 300)\n           .apply_filter("sharpen")\n           .save("output.png"))\n"""\nimport io\nfrom pathlib import Path\nfrom PIL import Image, ImageFilter, ImageEnhance\n\n_FILTERS = {\n    "blur":         ImageFilter.BLUR,\n    "sharpen":      ImageFilter.SHARPEN,\n    "edge_enhance": ImageFilter.EDGE_ENHANCE,\n    "contour":      ImageFilter.CONTOUR,\n}\n\n\nclass ImageProcessor:\n    """Chainable image processing pipeline.\n\n    Every mutating method returns self so calls can be chained.\n    """\n\n    def __init__(self, img: Image.Image) -> None:\n        self._img = img.copy()\n\n    @classmethod\n    def from_file(cls, path: str) -> "ImageProcessor":\n        return cls(Image.open(path))\n\n    @classmethod\n    def new(cls, width: int, height: int,\n            color: tuple = (255, 255, 255)) -> "ImageProcessor":\n        return cls(Image.new("RGB", (width, height), color=color))\n\n    @property\n    def image(self) -> Image.Image:\n        return self._img.copy()\n\n    @property\n    def size(self) -> tuple:\n        return self._img.size\n\n    @property\n    def mode(self) -> str:\n        return self._img.mode\n\n    def resize(self, width: int, height: int) -> "ImageProcessor":\n        self._img = self._img.resize((width, height), Image.Resampling.LANCZOS)\n        return self\n\n    def crop_center(self, width: int, height: int) -> "ImageProcessor":\n        iw, ih = self._img.size\n        left = (iw - width) // 2\n        top  = (ih - height) // 2\n        self._img = self._img.crop((left, top, left + width, top + height))\n        return self\n\n    def to_grayscale(self) -> "ImageProcessor":\n        self._img = self._img.convert("L")\n        return self\n\n    def convert_mode(self, mode: str) -> "ImageProcessor":\n        self._img = self._img.convert(mode)\n        return self\n\n    def apply_filter(self, filter_name: str) -> "ImageProcessor":\n        f = _FILTERS.get(filter_name.lower())\n        if f is None:\n            raise ValueError(\n                f"Unknown filter: {filter_name!r}. Available: {list(_FILTERS)}"\n            )\n        self._img = self._img.filter(f)\n        return self\n\n    def adjust_brightness(self, factor: float) -> "ImageProcessor":\n        self._img = ImageEnhance.Brightness(self._img).enhance(factor)\n        return self\n\n    def adjust_contrast(self, factor: float) -> "ImageProcessor":\n        self._img = ImageEnhance.Contrast(self._img).enhance(factor)\n        return self\n\n    def save(self, path: str, **kwargs) -> str:\n        p = Path(path)\n        p.parent.mkdir(parents=True, exist_ok=True)\n        out = self._img\n        if p.suffix.lower() in (".jpg", ".jpeg") and self._img.mode in ("RGBA", "P"):\n            out = self._img.convert("RGB")\n        out.save(path, **kwargs)\n        return path\n\n    def to_bytes(self, format: str = "PNG") -> bytes:\n        buf = io.BytesIO()\n        out = self._img\n        if format.upper() in ("JPEG", "JPG") and self._img.mode in ("RGBA", "P"):\n            out = self._img.convert("RGB")\n        out.save(buf, format=format)\n        return buf.getvalue()\n'
from pathlib import Path
Path('image_utils.py').write_text(_UTILS_SRC, encoding='utf-8')
print('image_utils.py written.')

In [ ]:
import os
import tempfile
from pathlib import Path
from PIL import Image
import numpy as np
from image_utils import ImageProcessor

# 1. Create image
p = ImageProcessor.new(200, 150, color=(100, 150, 200))
assert p.size == (200, 150), f"Expected (200, 150), got {p.size}"
assert p.mode == 'RGB'
print("\u2705 ImageProcessor.new creates correct image")

# 2. Resize (chainable)
p.resize(100, 75)
assert p.size == (100, 75), f"Expected (100, 75), got {p.size}"
print("\u2705 resize works")

# 3. Crop center
p2 = ImageProcessor.new(200, 200, color=(255, 0, 0))
p2.crop_center(100, 100)
assert p2.size == (100, 100), f"Expected (100, 100), got {p2.size}"
print("\u2705 crop_center works")

# 4. Grayscale
p3 = ImageProcessor.new(100, 100, color=(200, 100, 50))
p3.to_grayscale()
assert p3.mode == 'L', f"Expected L mode, got {p3.mode}"
print("\u2705 to_grayscale works")

# 5. Filter
p4 = ImageProcessor.new(100, 100, color=(200, 100, 50))
p4.apply_filter('blur')
assert p4.size == (100, 100)
print("\u2705 apply_filter works")

# 6. Unknown filter raises ValueError
raised = False
try:
    ImageProcessor.new(50, 50).apply_filter('teleport')
except ValueError:
    raised = True
assert raised
print("\u2705 unknown filter raises ValueError")

# 7. Chain operations
p5 = (
    ImageProcessor.new(400, 300, color=(50, 100, 150))
    .resize(200, 150)
    .apply_filter('sharpen')
    .adjust_brightness(1.1)
    .adjust_contrast(1.2)
)
assert p5.size == (200, 150), f"Expected (200, 150), got {p5.size}"
print("\u2705 method chaining works")

# 8. to_bytes returns valid PNG
raw = ImageProcessor.new(50, 50).to_bytes('PNG')
assert raw[:4] == b'\x89PNG', f"Not a PNG: {raw[:4]!r}"
print("\u2705 to_bytes returns valid PNG")

# 9. to_bytes JPEG — RGBA auto-converts
rgba_proc = ImageProcessor(Image.new('RGBA', (50, 50), (0, 200, 100, 128)))
jpeg_raw = rgba_proc.to_bytes('JPEG')
assert jpeg_raw[:2] == b'\xff\xd8'
print("\u2705 RGBA image converts to JPEG bytes without error")

# 10. save to disk
with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
    tmp_path = f.name
try:
    ImageProcessor.new(50, 50, color=(0, 255, 0)).save(tmp_path)
    assert Path(tmp_path).stat().st_size > 0
    print("\u2705 save to disk works")
finally:
    os.unlink(tmp_path)

print("\nImages in Python complete!")
